In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Ler o ficheiro
df = pd.read_csv('/Users/rr/Downloads/SAL_DAT027.csv')
df.shape
df = pd.read_csv('/Users/rr/Downloads/SAL_DAT027.csv')
df = df[df['GESTION'] == 'LIS']

In [2]:
import pandas as pd


df['CODACT'] = pd.to_numeric(df['CODACT'], errors='coerce')
df_lis = df[(df['GESTION'] == 'LIS') & (df['CODACT'] != 13)].copy()

df_lis['FENTREGA'] = pd.to_datetime(df_lis['FENTREGA'], format='%Y%m%d', errors='coerce')
df_lis['dia_semana'] = df_lis['FENTREGA'].dt.day_name()

# Dias da semana por referência
dias_por_ref = df_lis.groupby('REFERENCIA')['dia_semana'].apply(lambda x: sorted(set(x))).reset_index()
dias_por_ref.columns = ['REFERENCIA', 'dias_semana_distintos']
dias_por_ref['num_dias_distintos'] = dias_por_ref['dias_semana_distintos'].apply(len)

multiplos = dias_por_ref[dias_por_ref['num_dias_distintos'] > 1]

# Adicionar CODACT
multiplos_info = multiplos.merge(
    df_lis[['REFERENCIA', 'CODACT']].drop_duplicates(),
    on='REFERENCIA'
)

# Contar por padrão de dias
multiplos_info['dias_pattern'] = multiplos_info['dias_semana_distintos'].apply(str)
resumo_pattern = multiplos_info['dias_pattern'].value_counts()

print("Padrão de dias (quantas referências):")
resumo_pattern

Padrão de dias (quantas referências):


dias_pattern
['Monday', 'Saturday']       297
['Monday', 'Tuesday']         15
['Thursday', 'Wednesday']      1
Name: count, dtype: int64

In [3]:
# Por CODACT
resumo_codact = multiplos_info.groupby('CODACT').size()

print("\nPor CODACT:")
resumo_codact


Por CODACT:


CODACT
11.0      62
74.0      71
117.0      1
131.0      5
260.0      2
262.0      1
282.0      1
311.0    123
338.0      1
403.0      5
405.0      1
429.0     24
546.0      1
660.0      1
680.0     13
757.0      1
dtype: int64

In [4]:
multiplos_refs = multiplos['REFERENCIA'].tolist()

colunas = ['REFERENCIA', 'FENTREGA', 'dia_semana', 'LOCORIGEN', 'PROV_ORIGEN', 'LOCDESTINO', 'PROV_DESTINO', 
           'ENTREGAR', 'PROV_ENTREGAR', 'CODACT', 'ACTIVIDAD', 'PALETSDT', 'PESO_BRUTO', 'INGRESODT', 'COSTEDT']

resultado = df_lis[df_lis['REFERENCIA'].isin(multiplos_refs)][colunas].sort_values(['REFERENCIA', 'FENTREGA'])

resultado

,REFERENCIA,FENTREGA,dia_semana,LOCORIGEN,PROV_ORIGEN,LOCDESTINO,PROV_DESTINO,ENTREGAR,PROV_ENTREGAR,CODACT,ACTIVIDAD,PALETSDT,PESO_BRUTO,INGRESODT,COSTEDT
20424,0000024631,2026-08-01,Saturday,Azambuja,Lisboa,Modivas,Porto,H3 - Mar Shopping,Porto,429.0,H-3 Carnes,0.60,270.00,46.71,15.43
25867,0000024631,2026-08-03,Monday,Modivas,Porto,Porto,Porto,H3 - Mar Shopping,Porto,429.0,H-3 Carnes,0.60,270.00,46.71,32.93
20423,0000024632,2026-08-01,Saturday,Azambuja,Lisboa,Modivas,Porto,H3 CC ARRABIDA SHOPPING - AFURADA,Porto,429.0,H-3 Carnes,0.57,255.00,44.12,14.65
25891,0000024632,2026-08-03,Monday,Modivas,Porto,Vila Nova de Gaia,Porto,H3 CC ARRABIDA SHOPPING - AFURADA,Porto,429.0,H-3 Carnes,0.57,255.00,44.12,42.22
20425,0000024633,2026-08-01,Saturday,Azambuja,Lisboa,Modivas,Porto,H3 - NorteShopping,Porto,429.0,H-3 Carnes,0.90,405.00,70.07,23.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66949,REIS E PACHECO 17082026,2026-08-17,Monday,Modivas,Porto,Valongo,Porto,REIS & PACHECO,Porto,403.0,CARMONTI,3.00,0.00,74.13,129.09
80984,REIS E PACHECO 24082026,2026-08-22,Saturday,Montijo,Setubal,Modivas,Porto,REIS & PACHECO,Porto,403.0,CARMONTI,2.00,0.00,49.42,110.69
87879,REIS E PACHECO 24082026,2026-08-24,Monday,Modivas,Porto,Valongo,Porto,REIS & PACHECO,Porto,403.0,CARMONTI,2.00,0.00,49.42,56.78
46881,SECO-G.702312-377,2026-08-10,Monday,Modivas,Porto,São Mamede de Infesta,Porto,MAKRO - S. MAMEDE INFESTA,Porto,405.0,LACTACORES,8.00,6167.46,168.24,243.35
